# Training dataset review
This notebook builds one symbol dataset through production code, reports dataset readiness, and only reviews chronological splitting and training-only scaling when the production eligibility gate is satisfied. Insufficient local history is an expected readiness outcome, not an execution failure.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from data_pipeline.src.config import (
    AI_MINIMUM_USABLE_ROWS,
    MASTER_CSV_PATH,
    PROCESSED_SYMBOLS_DIR,
)
from feature_engineering.dataset_builder import build_symbol_datasets
from feature_engineering.preprocessing import fit_training_scaler
from feature_engineering.readiness import summarize_symbol_build_readiness
from feature_engineering.splitting import chronological_split

selected_symbol = "MCB"
minimum_usable_rows = AI_MINIMUM_USABLE_ROWS
dataset_path = PROCESSED_SYMBOLS_DIR / f"{selected_symbol}.csv"

raw_market = pd.read_csv(MASTER_CSV_PATH, dtype={"symbol": "string"})
symbol_history = raw_market.loc[
    raw_market["symbol"].astype("string").str.strip() == selected_symbol
].copy()
metrics = build_symbol_datasets(
    symbols=[selected_symbol],
    minimum_usable_rows=minimum_usable_rows,
)
readiness_summary = summarize_symbol_build_readiness(
    symbol=selected_symbol,
    raw_history=symbol_history,
    metrics=metrics,
    minimum_usable_rows=minimum_usable_rows,
    processed_path=dataset_path,
)
is_training_ready = readiness_summary.is_training_ready
readiness = readiness_summary.to_display_frame()

if is_training_ready:
    display(Markdown(f"### {selected_symbol} is ready for dataset review."))
else:
    display(Markdown(f"### ⚠️ {selected_symbol} is not ready for model training."))
display(readiness)
display(metrics.to_dict())

if not is_training_ready:
    display(
        Markdown(
            "Technical indicators require an initial warm-up period before rows "
            "become usable. The production dataset must then contain enough usable "
            "history for chronological training, validation, and test partitions. "
            "Collect more trading history and rebuild; no threshold was lowered and "
            "no rows were fabricated."
        )
    )

In [ ]:
dataset = None
split = None

if is_training_ready:
    dataset = pd.read_csv(dataset_path, dtype={"symbol": "string"})
    split = chronological_split(dataset, scope="symbol")
    display(split.metadata)
else:
    display(
        Markdown(
            f"**Chronological splitting skipped:** {selected_symbol} does not yet "
            "meet the production dataset-readiness requirements."
        )
    )

In [ ]:
if split is not None:
    train_dates = set(pd.to_datetime(split.train["date"]))
    validation_dates = set(pd.to_datetime(split.validation["date"]))
    test_dates = set(pd.to_datetime(split.test["date"]))
    assert train_dates.isdisjoint(validation_dates | test_dates)
    assert validation_dates.isdisjoint(test_dates)
    assert max(train_dates) < min(validation_dates) < min(test_dates)
    display(Markdown("**No date overlap or future leakage detected.**"))
else:
    display(Markdown("Overlap assertions skipped because no split was created."))

In [ ]:
scaled = None

if split is not None:
    scaled = fit_training_scaler(split.train, split.validation, split.test)
    display(
        pd.DataFrame(
            {
                "feature": scaled.scaled_features,
                "training_mean_before_scaling": scaled.scaler.mean_,
                "training_scale": scaled.scaler.scale_,
            }
        ).head(20)
    )
else:
    display(Markdown("Scaler review skipped because no split was created."))

In [ ]:
if split is not None and scaled is not None:
    comparison = pd.DataFrame(
        {
            "unscaled_close": split.validation["close"].head().to_numpy(),
            "scaled_close": scaled.validation["close"].head().to_numpy(),
        }
    )
    display(comparison)
else:
    display(Markdown("Scaled-value comparison skipped because no split was created."))

When the symbol is ready, the scaler is fitted only on `split.train`; validation and test partitions call `transform` with that fixed training scaler. Symbol/date/status identity columns are not scaled. When it is not ready, the guarded cells above intentionally perform no splitting, assertions, or scaling.